### Load file dataset từ formatted_jobs.csv


#### Việc cần làm
1. Load data
2. Clean text
3. Tạo combined_text cho từng job
4. Group theo Industry (chỉ để build vocab)

5. Với từng Industry:
   a. tokenize
   b. remove stopwords (custom)
   c. build frequency

6. Build word families:
   - stem → tạo group
   - remove outlier nhẹ

7. Chọn canonical word:
   - theo frequency
   - giữ word thật (không dùng stem)

8. Build mapping:
   activity → active

9. Rewrite từng job (KHÔNG phải industry text)

10. Vector hóa (TF-IDF)

11. Matching job ↔ candidate

In [2]:
import numpy as np
import pandas as pd
import re


In [3]:
def load_data(file_path): 
    df = pd.read_csv(file_path)
    required_cols = ['job_title', 'Short_description', 'Skills_required', 'Industry', 'Pay_grade']
    missing = [col for col in required_cols if col not in df.columns]
    if missing: 
        raise ValueError(f"Dữ liệu thiếu cột: {', '.join(missing)}")
    df= df.copy(
        
    )
    df = df.dropna(subset=['Industry']) 
    return df


In [6]:
df = load_data(r'E:\Semester6\Laptrinhpython\project\data\formatted_jobs.csv')
df.head()
df.info()
# kieerm tra cos bao nhieu loaij industry khac nhau
print("Tổng Industry: ")
print(df['Industry'].nunique())

<class 'pandas.DataFrame'>
RangeIndex: 970 entries, 0 to 969
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   ID_num             970 non-null    int64
 1   job_title          970 non-null    str  
 2   Short_description  970 non-null    str  
 3   Skills_required    970 non-null    str  
 4   Industry           970 non-null    str  
 5   Pay_grade          970 non-null    str  
dtypes: int64(1), str(5)
memory usage: 45.6 KB
Tổng Industry: 
105


In [ ]:
# 1. Thống kê số lượng dòng cho mỗi Industry
industry_counts = df['Industry'].value_counts()

print("Thống kê số lượng công việc theo từng ngành nghề:")
print(industry_counts)

# 2. (Tùy chọn) Nếu bạn muốn xem dưới dạng phần trăm (%)
industry_percent = df['Industry'].value_counts(normalize=True) * 100
print("\nPhần trăm tỷ trọng của từng ngành nghề:")
print(industry_percent)



Thống kê số lượng công việc theo từng ngành nghề:
Industry
Technology         197
Marketing           94
Healthcare          56
Education           36
Human Resources     35
                  ... 
Fishing              1
Food Services        1
Energy Services      1
Arts & Sciences      1
Energy               1
Name: count, Length: 105, dtype: int64

Phần trăm tỷ trọng của từng ngành nghề:
Industry
Technology         20.309278
Marketing           9.690722
Healthcare          5.773196
Education           3.711340
Human Resources     3.608247
                     ...    
Fishing             0.103093
Food Services       0.103093
Energy Services     0.103093
Arts & Sciences     0.103093
Energy              0.103093
Name: proportion, Length: 105, dtype: float64


In [8]:
# 1. Gộp các cột văn bản lại để tính toán vocab (loại bỏ giá trị NaN nếu có)
text_columns = ['job_title', 'Short_description', 'Skills_required']
df['all_text'] = df[text_columns].fillna('').agg(' '.join, axis=1)

# 2. Hàm định nghĩa cách tính số lượng từ duy nhất (Vocab size)
def count_unique_vocab(text_series):
    # Kết hợp toàn bộ văn bản của tất cả các dòng trong nhóm thành 1 chuỗi lớn
    full_content = " ".join(text_series).lower()
    # Sử dụng Regex để chỉ lấy các ký tự chữ và số (loại bỏ dấu câu)
    words = re.findall(r'\w+', full_content)
    # Trả về số lượng từ không trùng lặp
    return len(set(words))

# 3. Gom nhóm theo Industry và thực hiện tính toán đồng thời
# 'size' để đếm số dòng, 'count_unique_vocab' là hàm tự định nghĩa ở trên
summary_df = df.groupby('Industry').agg(
    So_luong_dong=('Industry', 'size'),
    Vocab_Size=('all_text', count_unique_vocab)
).reset_index()

# 4. Sắp xếp kết quả theo số lượng dòng giảm dần (tùy chọn)
summary_df = summary_df.sort_values(by='So_luong_dong', ascending=False)

# 5. Xuất ra file CSV
# 'utf-8-sig' giúp file CSV hiển thị đúng tiếng Việt khi mở bằng Excel
summary_df.to_csv('industry_stats.csv', index=False, encoding='utf-8-sig')

print("Đã tạo file 'industry_stats.csv' thành công!")
print(summary_df.head())

Đã tạo file 'industry_stats.csv' thành công!
           Industry  So_luong_dong  Vocab_Size
97       Technology            197         585
66        Marketing             94         219
51       Healthcare             56         324
25        Education             36         212
53  Human Resources             35         183


In [ ]:
SPECIAL_TOKEN_MAP = {
    "c++": "cpp",
    "c#": "csharp",
    ".net": "dotnet",
    "node.js": "nodejs",
    "react.js": "reactjs",
    "next.js": "nextjs",
    "vue.js": "vuejs",
    "nuxt.js": "nuxtjs",
    "express.js": "expressjs",
    "asp.net": "aspdotnet",
    "a.i": "ai",
    "m.l": "ml",
    "ui/ux": "uiux",
    "front-end": "frontend",
    "back-end": "backend",
    "full-stack": "fullstack",
    "data-science": "data_science",
    "machine-learning": "machine_learning",
    "deep-learning": "deep_learning",
    "problem-solving": "problem_solving",
    "decision-making": "decision_making",
}

def normalize_special_tokens(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()

    # normalize khoảng trắng trước
    text = re.sub(r"\s+", " ", text).strip()

    # thay các cụm đặc biệt theo thứ tự từ dài tới ngắn
    for src in sorted(SPECIAL_TOKEN_MAP.keys(), key=len, reverse=True):
        dst = SPECIAL_TOKEN_MAP[src]
        text = text.replace(src, dst)

    return text

In [10]:
def cleanText(text):
    if pd.isna(text):
        return ""

    text = normalize_special_tokens(text)

    # bỏ các ký tự không cần, nhưng tạm vẫn giữ _, +, #, .
    text = re.sub(r"[^\w\s\+\#\.]", " ", text, flags=re.UNICODE)

    # loại các dấu chấm dư còn sót lại nếu không còn ý nghĩa
    text = re.sub(r"\.(?=\s|$)", " ", text)

    # bỏ + và # đứng một mình, nhưng token đã normalize như cpp/csharp thì không bị ảnh hưởng
    text = re.sub(r"(?<!\w)[\+\#]+(?!\w)", " ", text)

    # chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [11]:
def formatTextColumns(df):
    df = df.copy()

    text_cols = ["job_title", "Short_description", "Skills_required", "Industry"]

    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").apply(cleanText)

    return df

# hàm này áp dụng cleanText cho tất cả các cột văn bản trong DataFrame

In [12]:
formatted_df = formatTextColumns(df)
formatted_df.head()

,ID_num,job_title,Short_description,Skills_required,Industry,Pay_grade
0,1,software engineer,develop and maintain web applications using mo...,problem solving logical reasoning attention to...,technology,High paying
1,2,data scientist,analyze large datasets to extract business ins...,analytical thinking pattern recognition mathem...,technology,High paying
2,3,marketing manager,lead marketing campaigns and brand strategy de...,creative thinking strategic planning communica...,marketing,Average paying
3,4,ux designer,design user friendly interfaces and improve us...,creative problem solving empathy research skil...,technology,Average paying
4,5,financial analyst,analyze financial data and prepare reports for...,analytical thinking attention to detail mathem...,finance,Average paying


In [13]:
def buildCombinedText(df):
    df = df.copy()

    df["combined_text"] = (
        df["job_title"].fillna("") + " " +
        df["Short_description"].fillna("") + " " +
        df["Skills_required"].fillna("")
    )

    df["combined_text"] = df["combined_text"].apply(lambda x: re.sub(r"\s+", " ", x).strip()) # hàm này chuẩn hóa khoảng trắng trong combined_text

    return df

In [15]:
built_df = buildCombinedText(formatted_df)
built_df = built_df[["Industry", "combined_text"]]
built_df.head()


,Industry,combined_text
0,technology,software engineer develop and maintain web app...
1,technology,data scientist analyze large datasets to extra...
2,marketing,marketing manager lead marketing campaigns and...
3,technology,ux designer design user friendly interfaces an...
4,finance,financial analyst analyze financial data and p...


In [16]:
import re
from collections import Counter, defaultdict
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

EN_STOPWORDS = set(stopwords.words("english"))

def tokenize_text(text):
    if pd.isna(text):
        return []
    
    text = str(text).strip().lower()
    
    # lấy token gồm chữ, số, underscore
    tokens = re.findall(r"\b[a-zA-Z0-9_]+\b", text)
    return tokens


def remove_stopwords(tokens, stop_words=EN_STOPWORDS, min_len=2):
    cleaned_tokens = [
        t for t in tokens
        if t not in stop_words and len(t) >= min_len
    ]
    return cleaned_tokens


def tokenize_and_remove_stopwords(text, stop_words=EN_STOPWORDS, min_len=2):
    tokens = tokenize_text(text)
    tokens = remove_stopwords(tokens, stop_words=stop_words, min_len=min_len)
    return tokens
# xây dựng token cho cột combined_text , loại bỏ stopwords

In [17]:
def build_vocabulary_by_industry(df, text_col="combined_text", industry_col="Industry"):
    vocab_by_industry = {}
    tokens_by_industry = {}

    grouped = df.groupby(industry_col)

    for industry, group_df in grouped:
        all_tokens = []

        for text in group_df[text_col].fillna(""):
            tokens = tokenize_and_remove_stopwords(text)
            all_tokens.extend(tokens)

        freq = Counter(all_tokens)

        vocab_by_industry[industry] = freq
        tokens_by_industry[industry] = all_tokens

    return vocab_by_industry, tokens_by_industry

In [20]:
vocab_by_industry, tokens_by_industry = build_vocabulary_by_industry(built_df)

list(vocab_by_industry.keys())
tokens_by_industry.keys()

dict_keys(['advertising', 'aerospace', 'agriculture', 'animal sciences', 'animal services', 'architecture', 'art antiques', 'arts crafts', 'arts culture', 'arts sciences', 'automotive', 'aviation', 'biotechnology', 'business services', 'cleaning services', 'construction', 'consulting', 'corporate governance', 'cosmetics', 'creative services', 'cultural heritage', 'customer service', 'design services', 'earth sciences', 'economics', 'education', 'emergency services', 'energy', 'energy research', 'energy services', 'engineering', 'entertainment', 'environmental sciences', 'environmental services', 'event planning', 'executive leadership', 'facilities management', 'fashion', 'finance', 'financial services', 'fishing', 'fitness recreation', 'food production', 'food science', 'food service', 'food services', 'forensic science', 'forestry', 'funeral services', 'gaming', 'government', 'healthcare', 'hospitality', 'human resources', 'insurance', 'international trade', 'jewelry', 'landscaping',

In [21]:
stemmer = PorterStemmer()

def group_family_words(freq_dict):
    stem_groups = defaultdict(list)

    for word, count in freq_dict.items():
        stem = stemmer.stem(word)
        stem_groups[stem].append((word, count))

    return stem_groups
# gom các family words lại với nhau, ví dụ như "develop", "developer", "development" sẽ có cùng gốc là "develop" và được gom vào cùng một nhóm.
# stem_groups sẽ có dạng vd: {"develop": [("develop", 100), ("developer", 80), ("development", 60)], ...}


In [25]:
grouped_vocab_by_industry = {}
for industry, freq_dict in vocab_by_industry.items():
    grouped_vocab_by_industry[industry] = group_family_words(freq_dict)
print(grouped_vocab_by_industry["technology"])

defaultdict(<class 'list'>, {'softwar': [('software', 20)], 'engin': [('engineer', 13), ('engineering', 14)], 'develop': [('develop', 12), ('developer', 13), ('development', 19)], 'maintain': [('maintain', 9)], 'web': [('web', 4)], 'applic': [('applications', 12), ('application', 2)], 'use': [('using', 3), ('use', 3)], 'modern': [('modern', 1)], 'framework': [('frameworks', 2)], 'problem': [('problem', 42)], 'solv': [('solving', 42)], 'logic': [('logical', 5)], 'reason': [('reasoning', 8)], 'attent': [('attention', 14)], 'detail': [('detail', 14)], 'team': [('team', 11), ('teams', 14)], 'collabor': [('collaboration', 3)], 'critic': [('critical', 6)], 'think': [('thinking', 14)], 'data': [('data', 35)], 'scientist': [('scientist', 2)], 'analyz': [('analyze', 14)], 'larg': [('large', 2)], 'dataset': [('datasets', 1)], 'extract': [('extract', 1)], 'busi': [('business', 4)], 'insight': [('insights', 9)], 'build': [('build', 6), ('building', 5)], 'predict': [('predictive', 1)], 'model': [('

In [27]:
def choose_canonical_word(stem_groups):
    family_mapping = {}
    canonical_info = {}

    for stem, items in stem_groups.items():
        # sort theo:
        # 1. frequency giảm dần
        # 2. độ dài tăng dần
        # 3. alphabet
        items_sorted = sorted(items, key=lambda x: (-x[1], len(x[0]), x[0]))
        canonical = items_sorted[0][0]

        canonical_info[stem] = {
            "canonical": canonical,
            "variants": items_sorted
        }

        for word, _ in items:
            family_mapping[word] = canonical

    return family_mapping, canonical_info

In [28]:
canonical_mapping_by_industry = {}
for industry, stem_groups in grouped_vocab_by_industry.items():
    mapping, info = choose_canonical_word(stem_groups)
    canonical_mapping_by_industry[industry] = mapping
print(canonical_mapping_by_industry["technology"])

{'software': 'software', 'engineer': 'engineering', 'engineering': 'engineering', 'develop': 'development', 'developer': 'development', 'development': 'development', 'maintain': 'maintain', 'web': 'web', 'applications': 'applications', 'application': 'applications', 'using': 'use', 'use': 'use', 'modern': 'modern', 'frameworks': 'frameworks', 'problem': 'problem', 'solving': 'solving', 'logical': 'logical', 'reasoning': 'reasoning', 'attention': 'attention', 'detail': 'detail', 'team': 'teams', 'teams': 'teams', 'collaboration': 'collaboration', 'critical': 'critical', 'thinking': 'thinking', 'data': 'data', 'scientist': 'scientist', 'analyze': 'analyze', 'large': 'large', 'datasets': 'datasets', 'extract': 'extract', 'business': 'business', 'insights': 'insights', 'build': 'build', 'building': 'build', 'predictive': 'predictive', 'models': 'models', 'modeling': 'models', 'model': 'models', 'analytical': 'analytics', 'analytics': 'analytics', 'pattern': 'pattern', 'patterns': 'pattern'

In [29]:
def normalize_vocabulary(freq_dict, family_mapping):
    normalized_freq = Counter()

    for word, count in freq_dict.items():
        canonical = family_mapping.get(word, word)
        normalized_freq[canonical] += count

    return normalized_freq

In [30]:
normalized_vocab_by_industry = {}
for industry, freq_dict in vocab_by_industry.items():
    family_mapping = canonical_mapping_by_industry[industry]
    normalized_freq = normalize_vocabulary(freq_dict, family_mapping)
    normalized_vocab_by_industry[industry] = normalized_freq
print(normalized_vocab_by_industry["technology"])

Counter({'product': 170, 'communication': 161, 'digital': 160, 'technology': 117, 'management': 78, 'remote': 67, 'analytics': 61, 'lead': 58, 'online': 53, 'technical': 51, 'design': 50, 'user': 50, 'reporting': 49, 'development': 44, 'problem': 42, 'solving': 42, 'analyst': 38, 'systems': 36, 'leadership': 36, 'data': 35, 'experience': 32, 'strategy': 30, 'support': 28, 'engineering': 27, 'customer': 26, 'teams': 25, 'security': 24, 'organization': 24, 'ux': 23, 'content': 23, 'documentation': 23, 'testing': 22, 'programming': 21, 'tools': 21, 'software': 20, 'innovation': 20, 'training': 18, 'analysis': 18, 'research': 17, 'growth': 17, 'skills': 16, 'create': 16, 'learning': 15, 'applications': 14, 'attention': 14, 'detail': 14, 'thinking': 14, 'analyze': 14, 'accessibility': 14, 'project': 14, 'writing': 14, 'ai': 13, 'cybersecurity': 12, 'help': 12, 'engagement': 12, 'campaign': 12, 'partnerships': 12, 'build': 11, 'creativity': 11, 'troubleshooting': 11, 'oversee': 11, 'service'

In [31]:
def normalize_vocabulary_by_industry(vocab_by_industry):
    normalized_vocab_by_industry = {}
    mapping_by_industry = {}
    canonical_info_by_industry = {}

    for industry, freq_dict in vocab_by_industry.items():
        stem_groups = group_family_words(freq_dict)
        family_mapping, canonical_info = choose_canonical_word(stem_groups)
        normalized_freq = normalize_vocabulary(freq_dict, family_mapping)

        normalized_vocab_by_industry[industry] = normalized_freq
        mapping_by_industry[industry] = family_mapping
        canonical_info_by_industry[industry] = canonical_info

    return normalized_vocab_by_industry, mapping_by_industry, canonical_info_by_industry
# output sẽ có dạng:
# normalized_vocab_by_industry = {
#     "Software": Counter({"develop": 240, "test": 150, ...}),
#     "Finance": Counter({"analyz": 180, "report": 120, ...}),
#     ...
# },
# mapping_by_industry = {
#     "Software": {"develop": "develop", "developer": "develop", "development": "develop", ...},
#     "Finance": {"analyz": "analyz", "analyze":
#         "analyz", "report": "report", ...},
#     ...
# },
# canonical_info_by_industry = {
#     "Software": {
#         "develop": {
#             "canonical": "develop",
#             "variants": [("develop", 100), ("developer", 80), ("development", 60)]
#         },
#         ...
#     },
#     ...
# }


In [33]:
normalized_vocab_by_industry = {}
for industry, freq_dict in vocab_by_industry.items():
    family_mapping = canonical_mapping_by_industry[industry]
    normalized_freq = normalize_vocabulary(freq_dict, family_mapping)
    normalized_vocab_by_industry[industry] = normalized_freq
print(normalized_vocab_by_industry["technology"])

Counter({'product': 170, 'communication': 161, 'digital': 160, 'technology': 117, 'management': 78, 'remote': 67, 'analytics': 61, 'lead': 58, 'online': 53, 'technical': 51, 'design': 50, 'user': 50, 'reporting': 49, 'development': 44, 'problem': 42, 'solving': 42, 'analyst': 38, 'systems': 36, 'leadership': 36, 'data': 35, 'experience': 32, 'strategy': 30, 'support': 28, 'engineering': 27, 'customer': 26, 'teams': 25, 'security': 24, 'organization': 24, 'ux': 23, 'content': 23, 'documentation': 23, 'testing': 22, 'programming': 21, 'tools': 21, 'software': 20, 'innovation': 20, 'training': 18, 'analysis': 18, 'research': 17, 'growth': 17, 'skills': 16, 'create': 16, 'learning': 15, 'applications': 14, 'attention': 14, 'detail': 14, 'thinking': 14, 'analyze': 14, 'accessibility': 14, 'project': 14, 'writing': 14, 'ai': 13, 'cybersecurity': 12, 'help': 12, 'engagement': 12, 'campaign': 12, 'partnerships': 12, 'build': 11, 'creativity': 11, 'troubleshooting': 11, 'oversee': 11, 'service'

In [35]:
def build_vocab_list_by_industry(normalized_vocab_by_industry):
    vocab_list_by_industry = {}

    for industry, freq_dict in normalized_vocab_by_industry.items():
        vocab_list_by_industry[industry] = list(freq_dict.keys())

    return vocab_list_by_industry

In [36]:
vocab_list_by_industry = build_vocab_list_by_industry(normalized_vocab_by_industry)
print(vocab_list_by_industry["technology"])

['software', 'engineering', 'development', 'maintain', 'web', 'applications', 'use', 'modern', 'frameworks', 'problem', 'solving', 'logical', 'reasoning', 'attention', 'detail', 'teams', 'collaboration', 'critical', 'thinking', 'data', 'scientist', 'analyze', 'large', 'datasets', 'extract', 'business', 'insights', 'build', 'predictive', 'models', 'analytics', 'pattern', 'recognition', 'mathematics', 'research', 'skills', 'ux', 'design', 'user', 'friendly', 'interfaces', 'improve', 'experience', 'creativity', 'empathy', 'visual', 'centered', 'network', 'administrator', 'troubleshooting', 'computer', 'systems', 'technical', 'patience', 'devops', 'implement', 'ci', 'cd', 'pipelines', 'management', 'cloud', 'infrastructure', 'automation', 'mindset', 'continuous', 'learning', 'quality', 'assurance', 'tester', 'testing', 'identify', 'bug', 'ensure', 'systematic', 'websites', 'various', 'programming', 'languages', 'adapt', 'database', 'integrity', 'security', 'awareness', 'cybersecurity', 'an

In [37]:
def export_vocab_comma_format(normalized_vocab_by_industry, output_path="industry_vocab.csv"):
    rows = []

    for industry, freq_dict in normalized_vocab_by_industry.items():
        vocab_list = list(freq_dict.keys())

        rows.append({
            "Industry": industry,
            "vocab": ", ".join(vocab_list)
        })

    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    return df

In [38]:
export_df = export_vocab_comma_format(normalized_vocab_by_industry)
export_df.head()

,Industry,vocab
0,advertising,"advertising, executive, develop, implement, ca..."
1,aerospace,"space, mission, planner, design, coordinate, e..."
2,agriculture,"farmer, grow, crop, raise, livestock, food, pr..."
3,animal sciences,"animal, behaviorist, study, modify, behavior, ..."
4,animal services,"pet, grief, counselor, support, families, loss..."
